# CounterFeint — Investigator GRPO Training (single submission notebook)

**This is the entry-point notebook judges run.** It takes a fresh
`Qwen/Qwen2.5-1.5B-Instruct` (Apache 2.0, no HF gating), fine-tunes
the **Investigator** policy with TRL `GRPOTrainer` inside the
CounterFeint multi-agent FraudArena, then writes the headline
before-vs-after artefacts the README embeds:

```
eval_outputs/
├── eval_results.json   # per-episode metrics for both tags
├── eval_summary.md     # markdown delta table (before / after / delta)
└── eval_plot.png       # 2×2 bar chart, headline visual
```

Per the hackathon rules ("*A working training script using Unsloth or
Hugging Face TRL, ideally as a Colab notebook so judges can re-run
it.*") **all training code is inlined into this notebook** — there are
no helper Python files to import. Production-time policies
(`counterfeint.agents.LLMFraudster`, `counterfeint.scripted.*`) and
the eval harness (`counterfeint.eval_suite`) stay as repo modules
because they are also used at inference / serving time.

## The asymmetric multi-agent narrative

| Role         | Model                                  | Backend               | Trainable? |
|--------------|----------------------------------------|-----------------------|------------|
| Investigator | `Qwen/Qwen2.5-1.5B-Instruct`           | HuggingFace + QLoRA   | **Yes — this notebook** |
| Fraudster    | `llama3.1:8b`                          | Ollama (frozen)       | No         |
| Auditor      | Heuristic (regex + light scoring)      | In-process Python     | No         |

We deliberately train the **5× smaller, different-family** model
against the **larger** frozen adversary. The Llama 3.1 8B Fraudster
is a much stronger ad generator than the 1.5B Qwen Investigator at
baseline, and the headline result is "*the smaller specialised
Investigator beats the larger generalist Fraudster via task-specific
GRPO*" — exactly the kind of multi-agent / self-improvement story
themes #1 and #4 of the hackathon brief reward. Cross-family
(Qwen vs Llama) also rules out parameter-sharing as the source of the
GRPO signal.

Every Fraudster rollout uses the LLM (no scripted shortcut), so the
training distribution is genuinely adversarial and matches the eval
distribution. We pay for that with a longer training run; see the
`MODE` table below.

## Three runtime budgets — pick one in §1

| `MODE`  | Episodes | Eval seeds   | ~Wall time (24 GB GPU)    | What it proves |
|---------|---------:|-------------:|---------------------------|----------------|
| `smoke` |    6     | 3 (1/task)   | ~10 min (`dry_run=True`, GRPO step skipped) | Pipeline runs end-to-end; artefacts written. |
| `demo`  |   30     | 30 (10/task) | ~45–75 min                | Visible before/after delta; headline plot meaningful. |
| `full`  |  150     | 30 (10/task) | ~4–6 h                    | Convergence-quality curve. |

Episode counts are smaller than for a scripted-fraudster setup because
every rollout pays the 8B-Fraudster latency; the GRPO sample count
(samples ≈ episodes × ~5–10 turns) is still well above the
verifiable-reward GRPO floor.


## 1 · Configuration

Everything judges might want to tweak lives in **this single cell**.
Pick a `MODE`, optionally point `BASE_MODEL` at a different HF ID, then
run all cells below.


In [1]:
from pathlib import Path

MODE: str = "smoke"
assert MODE in {"smoke", "demo", "full"}, MODE

BASE_MODEL: str = "Qwen/Qwen2.5-1.5B-Instruct"

FRAUDSTER_OLLAMA_MODEL: str = "llama3.1:8b"
USE_OLLAMA_FRAUDSTER: bool = True
LLM_FRAUDSTER_RATIO: float = 1.0
LLM_FRAUDSTER_TIMEOUT_S: float = 120.0

SHOW_INTERACTION_TRACE: bool = True
TRACE_MAX_RATIONALE_CHARS: int = 80

TRAINED_TAG: str = "investigator_grpo_gen0"

OUT_DIR: Path = Path("training_outputs") / TRAINED_TAG
EVAL_OUT_DIR: Path = Path("eval_outputs")

TRAINING_SEED_TIERS = {
    "smoke": {
        "task_1": [0, 1],
        "task_2": [100, 101],
        "task_3": [200, 201],
    },
    "demo": {
        "task_1": list(range(0, 10)),
        "task_2": list(range(100, 110)),
        "task_3": list(range(200, 210)),
    },
    "full": {
        "task_1": list(range(0, 50)),
        "task_2": list(range(100, 150)),
        "task_3": list(range(200, 250)),
    },
}
EVAL_SEED_TIERS = {
    "smoke": {"task_1": [1001], "task_2": [2001], "task_3": [3001]},
    "demo": None,
    "full": None,
}

TRAINING_SEEDS = TRAINING_SEED_TIERS[MODE]
EVAL_SEEDS_OVERRIDE = EVAL_SEED_TIERS[MODE]
DRY_RUN = MODE == "smoke"

n_train_eps = sum(len(v) for v in TRAINING_SEEDS.values())
print(f"MODE                   = {MODE}")
print(f"BASE_MODEL             = {BASE_MODEL}")
print(f"FRAUDSTER_OLLAMA_MODEL = {FRAUDSTER_OLLAMA_MODEL}")
print(f"USE_OLLAMA_FRAUDSTER   = {USE_OLLAMA_FRAUDSTER}")
print(f"LLM_FRAUDSTER_RATIO    = {LLM_FRAUDSTER_RATIO}  (1.0 = 100% LLM, 0.0 = 100% scripted)")
print(f"LLM_FRAUDSTER_TIMEOUT_S= {LLM_FRAUDSTER_TIMEOUT_S}  (per-call timeout for the 8B Ollama model)")
print(f"SHOW_INTERACTION_TRACE = {SHOW_INTERACTION_TRACE}  (one line per agent turn during rollouts)")
print(f"OUT_DIR                = {OUT_DIR.resolve()}")
print(f"EVAL_OUT_DIR           = {EVAL_OUT_DIR.resolve()}")
print(f"Training episodes: {n_train_eps}  (across {len(TRAINING_SEEDS)} tasks)")
print(f"DRY_RUN                = {DRY_RUN}  (smoke mode skips GRPOTrainer.train)")


MODE                   = smoke
BASE_MODEL             = Qwen/Qwen2.5-1.5B-Instruct
FRAUDSTER_OLLAMA_MODEL = llama3.1:8b
USE_OLLAMA_FRAUDSTER   = True
LLM_FRAUDSTER_RATIO    = 1.0  (1.0 = 100% LLM, 0.0 = 100% scripted)
LLM_FRAUDSTER_TIMEOUT_S= 120.0  (per-call timeout for the 8B Ollama model)
SHOW_INTERACTION_TRACE = True  (one line per agent turn during rollouts)
OUT_DIR                = C:\Users\abhij\OneDrive\Desktop\Brain Labs\Coding\OpenEnv Hackathon\counterfeint\training\training_outputs\investigator_grpo_gen0
EVAL_OUT_DIR           = C:\Users\abhij\OneDrive\Desktop\Brain Labs\Coding\OpenEnv Hackathon\counterfeint\training\eval_outputs
Training episodes: 6  (across 3 tasks)
DRY_RUN                = True  (smoke mode skips GRPOTrainer.train)


## 2 · Environment setup

Three sub-steps:

1. Install runtime + training deps from `pyproject.toml` /
   `requirements-train.txt`.
2. (Optional) Authenticate with HuggingFace. The default `BASE_MODEL`
   is `Qwen/Qwen2.5-1.5B-Instruct` (Apache 2.0, no gating, no token
   needed). If you change `BASE_MODEL` to a gated model
   (e.g. `meta-llama/Llama-*`), either set `HF_TOKEN` in the
   environment / Colab secrets or paste it interactively here.
3. Print GPU info + register `counterfeint` on `sys.path`.

> **Personal Windows laptop with Device Guard?** Don't run
> `huggingface-cli login` in PowerShell — Device Guard blocks the
> exe. The Python `huggingface_hub.login(token=...)` call below uses
> only Python APIs (no subprocess, no exe). The simplest
> workaround is to set `$env:HF_TOKEN = "hf_xxx..."` in PowerShell
> before launching Jupyter, OR uncomment the explicit-token line at
> the top of the auth cell below (commit-safe — the cell deletes
> `_token` afterwards).


In [2]:
import os
import subprocess
import sys
from pathlib import Path

def _find_repo_root() -> Path:
    """Walk up from cwd / this notebook's dir until we find the counterfeint repo root.

    Robust to launching Jupyter from anywhere (`counterfeint/`,
    `counterfeint/training/`, the workspace parent, etc.).
    """
    candidates = [Path.cwd().resolve()]
    candidates.extend(candidates[0].parents)
    try:
        nb_dir = Path(__file__).resolve().parent
        candidates = [nb_dir, *nb_dir.parents, *candidates]
    except NameError:
        pass

    seen: set[Path] = set()
    for cand in candidates:
        if cand in seen:
            continue
        seen.add(cand)
        if (cand / "pyproject.toml").is_file() and (cand / "server").is_dir() and (cand / "agents").is_dir():
            return cand
    raise RuntimeError(
        "Could not locate the counterfeint/ repo root from\n"
        f"  cwd = {Path.cwd().resolve()}\n"
        "Looked for a directory containing all of: pyproject.toml, server/, agents/.\n"
        "Either launch Jupyter from inside the counterfeint/ folder, or open the\n"
        "notebook from a path inside the repo."
    )

_repo_root = _find_repo_root()
if Path.cwd().resolve() != _repo_root:
    print(f"Switching working directory:\n  {Path.cwd()}\n  -> {_repo_root}")
    os.chdir(_repo_root)

print(f"Repo root = {Path.cwd().resolve()}")

def _pip(*args: str) -> None:
    """Install into the SAME Python the kernel is using (avoids 'installed but not importable')."""
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    print(f"  $ {' '.join(cmd[1:])}")
    subprocess.run(cmd, check=True)

print(">> Installing CounterFeint package + dev deps ...")
_pip("-e", ".")
_pip("-r", "requirements-dev.txt")

print(">> Installing GRPO training stack (transformers / peft / trl / bitsandbytes / ...)")
_pip("-r", "requirements-train.txt")

print(">> Done.")


Switching working directory:
  C:\Users\abhij\OneDrive\Desktop\Brain Labs\Coding\OpenEnv Hackathon\counterfeint\training
  -> C:\Users\abhij\OneDrive\Desktop\Brain Labs\Coding\OpenEnv Hackathon\counterfeint
Repo root = C:\Users\abhij\OneDrive\Desktop\Brain Labs\Coding\OpenEnv Hackathon\counterfeint
>> Installing CounterFeint package + dev deps ...
  $ -m pip install -q -e .
  $ -m pip install -q -r requirements-dev.txt
>> Installing GRPO training stack (transformers / peft / trl / bitsandbytes / ...)
  $ -m pip install -q -r requirements-train.txt
>> Done.


In [3]:
import os

# --- PERSONAL-LAPTOP ESCAPE HATCH (Windows Device Guard friendly) ---
# Uncomment ONE line below if you cannot set HF_TOKEN as an env var
# (e.g. corporate Windows machine where Device Guard blocks
# huggingface-cli.exe). The cell deletes the variable after login so
# the token does not linger in the kernel's globals.
# os.environ["HF_TOKEN"] = "hf_REPLACE_ME"

_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")

if not _token:
    try:
        from google.colab import userdata
        _token = userdata.get("HF_TOKEN")
    except Exception:
        _token = None

from huggingface_hub import login

if _token:
    login(token=_token, add_to_git_credential=False)
    print("HuggingFace login: OK (token from env / Colab secret / inline)")
else:
    print("No HF_TOKEN found in env or Colab secrets — falling back to interactive login.")
    login()

del _token


No HF_TOKEN found in env or Colab secrets — falling back to interactive login.


In [4]:
import os
import sys
from pathlib import Path

if "_find_repo_root" in globals():
    REPO_ROOT = _find_repo_root()
else:
    REPO_ROOT = Path.cwd().resolve()

_looks_like_counterfeint = (
    (REPO_ROOT / "pyproject.toml").is_file()
    and (REPO_ROOT / "server").is_dir()
    and (REPO_ROOT / "agents").is_dir()
)
if not _looks_like_counterfeint:
    raise RuntimeError(
        f"REPO_ROOT={REPO_ROOT} does not look like the counterfeint/ root "
        f"(missing pyproject.toml + server/ + agents/).\n"
        f"Re-run §2 first - it auto-locates the repo and chdirs into it."
    )

if Path.cwd().resolve() != REPO_ROOT:
    os.chdir(REPO_ROOT)

if str(REPO_ROOT.parent) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT.parent))

os.environ.setdefault("COUNTERFEINT_ENV_URL", "http://localhost:8000")
os.environ.setdefault("API_BASE_URL", "http://localhost:11434/v1")
os.environ["MODEL_NAME"] = FRAUDSTER_OLLAMA_MODEL
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")

try:
    import nest_asyncio
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "nest_asyncio>=1.6.0"],
        check=True,
    )
    import nest_asyncio
nest_asyncio.apply()
print("nest_asyncio: applied (Jupyter/Colab-safe asyncio.run)")

try:
    import torch
    print(f"torch     : {torch.__version__}")
    print(f"CUDA avail: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU       : {torch.cuda.get_device_name(0)}")
        print(f"VRAM (GB) : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}")
        if torch.cuda.get_device_properties(0).total_memory < 10e9:
            print("WARNING   : <10 GB VRAM detected. Training the 1.5B Investigator")
            print("            with QLoRA AND running an 8B Fraudster via Ollama")
            print("            on the same GPU will OOM. Either use MODE='smoke'")
            print("            (skips the GRPO step) or set USE_OLLAMA_FRAUDSTER=False.")
    else:
        print("WARNING   : no CUDA device. Training will be extremely slow.")
        print("            Use MODE='smoke' (dry_run skips the actual TRL train call).")
except ImportError:
    print("torch not installed yet - the previous cell should have done that. Re-run it.")

print("\nEnvironment:")
for k in ("COUNTERFEINT_ENV_URL", "API_BASE_URL", "MODEL_NAME"):
    print(f"  {k} = {os.environ.get(k)}")


nest_asyncio: applied (Jupyter/Colab-safe asyncio.run)
FraudArena server: PID=16940  /health=200  log=C:\Users\abhij\OneDrive\Desktop\Brain Labs\Coding\OpenEnv Hackathon\counterfeint\server.log


## 3 · Start the FraudArena server (background)

The training loop talks to the OpenEnv `RefereeEnvironment` over HTTP
(stable, identical wire format to inference). We start it in the
background and wait for `/health` to respond before continuing.


In [5]:
import subprocess
import time
import urllib.request

def _server_reachable() -> bool:
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=1) as r:
            return r.status == 200
    except Exception:
        return False

SERVER_LOG = REPO_ROOT / "server.log"
server_proc = None

if _server_reachable():
    print(f"FraudArena already running on :8000 - reusing existing instance.  log={SERVER_LOG}")
else:
    _log_fp = open(SERVER_LOG, "w", buffering=1, encoding="utf-8")
    server_proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "counterfeint.server.app:app",
         "--host", "127.0.0.1", "--port", "8000",
         "--ws-ping-interval", "0",
         "--ws-ping-timeout", "0"],
        cwd=str(REPO_ROOT.parent),
        stdout=_log_fp,
        stderr=subprocess.STDOUT,
    )

_HEALTH_TIMEOUT_S = 60
deadline = time.time() + _HEALTH_TIMEOUT_S
ok = False
while time.time() < deadline:
    if _server_reachable():
        ok = True
        break
    if server_proc is not None and server_proc.poll() is not None:
        break
    time.sleep(0.5)

if ok:
    pid_str = f"PID={server_proc.pid}" if server_proc is not None else "PID=existing"
    print(f"FraudArena server: {pid_str}  /health=200  log={SERVER_LOG}")
else:
    log_tail = ""
    try:
        with open(SERVER_LOG, "r", encoding="utf-8", errors="replace") as fp:
            log_tail = "".join(fp.readlines()[-25:])
    except Exception:
        pass
    proc_status = (
        "process exited"
        if server_proc is not None and server_proc.poll() is not None
        else f"timed out after {_HEALTH_TIMEOUT_S}s"
    )
    raise RuntimeError(
        f"FraudArena server did not become healthy ({proc_status}).\n"
        f"Last 25 lines of {SERVER_LOG}:\n"
        f"{'-' * 60}\n{log_tail}{'-' * 60}"
    )


## 4 · Start Ollama for the frozen Llama 3.1 8B Fraudster

Every Fraudster rollout calls Ollama at `localhost:11434` to generate
ad copy + targeting + landing-page text. Set
`USE_OLLAMA_FRAUDSTER = False` in §1 (or `LLM_FRAUDSTER_RATIO = 0.0`)
to fall back to the deterministic `ReactiveFraudster` if you don't
have the VRAM headroom for both 1.5B-train + 8B-serve.

On a fresh Colab the install adds ~150 MB and the 8B pull is ~4.7 GB;
plan for 3–8 minutes the first time.


In [6]:
ollama_proc = None

def _ollama_reachable() -> bool:
    try:
        with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

if USE_OLLAMA_FRAUDSTER:
    import platform
    import shutil

    if not _ollama_reachable():
        if shutil.which("ollama") is None:
            if platform.system() == "Windows":
                raise RuntimeError(
                    "Ollama not found in PATH on Windows. Install it from "
                    "https://ollama.com/download/windows then re-run this cell. "
                    "(After the installer finishes, Ollama auto-starts as a tray app and "
                    "this cell will detect it.)"
                )
            elif platform.system() == "Darwin":
                raise RuntimeError(
                    "Ollama not found in PATH on macOS. Install it from "
                    "https://ollama.com/download/mac then re-run this cell."
                )
            else:
                print("Installing Ollama (Linux/Colab) ...")
                subprocess.run(
                    "curl -fsSL https://ollama.com/install.sh | sh",
                    shell=True, check=True,
                )

        OLLAMA_LOG = REPO_ROOT / "ollama.log"
        _ollama_log_fp = open(OLLAMA_LOG, "w", buffering=1, encoding="utf-8")
        ollama_proc = subprocess.Popen(
            ["ollama", "serve"], stdout=_ollama_log_fp, stderr=subprocess.STDOUT,
        )
        for _ in range(20):
            if _ollama_reachable():
                break
            time.sleep(0.5)
        else:
            raise RuntimeError(f"Ollama failed to start within 10s. See {OLLAMA_LOG}.")
        print(f"ollama serve: PID={ollama_proc.pid}  log={OLLAMA_LOG}")
    else:
        print("Ollama already running on :11434 - reusing existing instance.")

    print(f"Pulling {FRAUDSTER_OLLAMA_MODEL} (skipped if already pulled) ...")
    subprocess.run(["ollama", "pull", FRAUDSTER_OLLAMA_MODEL], check=True)
    print("Ollama ready.")
else:
    print("USE_OLLAMA_FRAUDSTER=False  -  every rollout will use ReactiveFraudster.")
    print("(That keeps training fast at the cost of weakening the multi-agent narrative.)")


ollama serve: PID=10596  log=C:\Users\abhij\OneDrive\Desktop\Brain Labs\Coding\OpenEnv Hackathon\counterfeint\ollama.log
Pulling llama3.1:8b (skipped if already pulled) ...
Ollama ready.


## 5 · Training stack — `HFInvestigator`

This is the policy whose weights GRPO updates. It wraps a local
`transformers` model directly (no HTTP hop) so TRL's `GRPOTrainer`
can backprop through it. The system prompt + JSON parsing are reused
verbatim from `counterfeint.agents.LLMInvestigator` so the trained
LoRA adapter slots straight back into the production HTTP-based
inference path after training.

Failure handling: any error (generation crash, JSON decode, schema
validation, OOM-safe truncation) routes to the deterministic
`ScriptedInvestigator`, increments `fallback_count`, and lets the
rollout finish so GRPO sees a full episode reward.


In [7]:
import json
import logging
import re
from typing import Any, Dict, Optional

from counterfeint.agents.prompts import (
    INVESTIGATOR_SYSTEM_PROMPT,
    INVESTIGATOR_USER_TEMPLATE,
)
from counterfeint.models import AdReviewAction
from counterfeint.scripted._base import PolicyBase
from counterfeint.scripted.investigator import ScriptedInvestigator

logger = logging.getLogger("counterfeint.notebook")
_JSON_FENCE_RE = re.compile(r"```(?:json)?\s*\n(.*?)```", re.DOTALL)


def _extract_json_text(raw: str) -> str:
    text = (raw or "").strip()
    m = _JSON_FENCE_RE.search(text)
    if m:
        return m.group(1).strip()
    if text.startswith("```"):
        lines = [l for l in text.split("\n") if not l.strip().startswith("```")]
        return "\n".join(lines).strip()
    return text


def _truncate(s: str, n: int) -> str:
    if not s:
        return ""
    s = s.strip()
    return s if len(s) <= n else s[: n - 3] + "..."


class HFInvestigator(PolicyBase):
    """Investigator policy backed by a local transformers model."""

    system_prompt = INVESTIGATOR_SYSTEM_PROMPT
    action_model = AdReviewAction
    _log_name = "hf_investigator"

    def __init__(
        self,
        *,
        model: Any,
        tokenizer: Any,
        fallback_policy: Optional[PolicyBase] = None,
        max_new_tokens: int = 384,
        temperature: float = 0.7,
        do_sample: bool = True,
    ) -> None:
        if model is None or tokenizer is None:
            raise TypeError(
                "HFInvestigator requires both `model` and `tokenizer`."
            )
        self.model = model
        self.tokenizer = tokenizer
        self.fallback_policy = fallback_policy or ScriptedInvestigator()
        self.max_new_tokens = int(max_new_tokens)
        self.temperature = float(temperature)
        self.do_sample = bool(do_sample)

        self.fallback_count = 0
        self.call_count = 0
        self.last_error: Optional[str] = None
        self.last_prompt: Optional[str] = None
        self.last_completion: Optional[str] = None

    @classmethod
    def from_pretrained(
        cls,
        model_name_or_path: str,
        *,
        load_in_4bit: bool = True,
        lora_path: Optional[str] = None,
        device_map: str = "auto",
        **kwargs: Any,
    ) -> "HFInvestigator":
        import torch  # noqa: F401
        from transformers import (
            AutoModelForCausalLM,
            AutoTokenizer,
            BitsAndBytesConfig,
        )

        tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        model_kwargs: Dict[str, Any] = {"device_map": device_map}
        if load_in_4bit:
            model_kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype="bfloat16",
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
            )
        model = AutoModelForCausalLM.from_pretrained(
            model_name_or_path, **model_kwargs
        )

        if lora_path:
            from peft import PeftModel
            model = PeftModel.from_pretrained(model, lora_path)

        return cls(model=model, tokenizer=tokenizer, **kwargs)

    def reset(self) -> None:
        self.fallback_count = 0
        self.call_count = 0
        self.last_error = None
        self.last_prompt = None
        self.last_completion = None
        if self.fallback_policy is not None:
            self.fallback_policy.reset()

    def act(self, observation: Dict[str, Any]) -> Any:
        self.call_count += 1
        try:
            user_prompt = self._build_user_prompt(observation)
            self.last_prompt = user_prompt
            raw = self._call_local_model(user_prompt)
            self.last_completion = raw
            return self._parse_and_validate(raw)
        except Exception as exc:  # noqa: BLE001
            self.fallback_count += 1
            self.last_error = f"{type(exc).__name__}: {exc}"
            logger.warning(
                "[HF-investigator] step %d failed (%s); delegating to %s",
                self.call_count, self.last_error,
                type(self.fallback_policy).__name__,
            )
            return self.fallback_policy.act(observation)

    def _build_user_prompt(self, observation: Dict[str, Any]) -> str:
        pending = observation.get("available_ads", []) or []
        queue_status = observation.get("queue_status") or {}
        findings = _truncate(
            observation.get("investigation_findings") or "(nothing pulled yet)",
            n=1800,
        )
        verdict_history = _truncate(
            observation.get("verdict_history_summary") or "(no verdicts yet)",
            n=600,
        )
        current_ad_info = observation.get("current_ad_info") or (
            "No ad in focus yet — focus one by starting with an investigate "
            "action on any pending ad."
        )
        return INVESTIGATOR_USER_TEMPLATE.format(
            task_id=queue_status.get("task_id", "?"),
            steps_remaining=queue_status.get("steps_remaining", "?"),
            investigation_budget=queue_status.get("investigation_budget", "?"),
            reviewed_count=queue_status.get("reviewed", 0),
            queue_may_grow=observation.get("queue_may_grow", False),
            pending_len=len(pending),
            pending_preview=", ".join(pending[:12])
            + (f" +{len(pending) - 12}" if len(pending) > 12 else ""),
            current_ad_info=current_ad_info,
            findings_preview=findings,
            verdict_history=verdict_history,
            feedback=(observation.get("feedback") or "").strip() or "(none)",
        )

    def _call_local_model(self, user_prompt: str) -> str:
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        encoded = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )
        try:
            device = next(self.model.parameters()).device
        except StopIteration:
            device = getattr(self.model, "device", None)
        if device is not None:
            encoded = {k: v.to(device) for k, v in encoded.items()}
        gen_kwargs = {
            "max_new_tokens": self.max_new_tokens,
            "temperature": self.temperature,
            "do_sample": self.do_sample,
            "pad_token_id": getattr(self.tokenizer, "pad_token_id", None)
            or getattr(self.tokenizer, "eos_token_id", 0),
        }
        outputs = self.model.generate(**encoded, **gen_kwargs)
        prompt_len = encoded["input_ids"].shape[-1]
        gen_tokens = outputs[0][prompt_len:]
        return self.tokenizer.decode(gen_tokens, skip_special_tokens=True)

    _ALLOWED_KEYS = {
        "action_type", "ad_id",
        "investigation_target",
        "verdict", "confidence", "rationale",
        "linked_ad_id", "link_reason",
    }
    _ALIAS_MAP = {
        "investigation_rationale": "rationale",
        "investigation_reason": "rationale",
        "investigation_confidence": "confidence",
        "linked_ads_id": "linked_ad_id",
        "link_account_reason": "link_reason",
    }
    _ALLOWED_TARGETS = {
        "advertiser_history", "landing_page", "payment_method",
        "targeting_overlap", "campaign_structure", "policy_classifier",
    }

    def _coerce_to_schema(self, data: Dict[str, Any]) -> Dict[str, Any]:
        out: Dict[str, Any] = {}
        for k, v in data.items():
            tgt = self._ALIAS_MAP.get(k, k)
            if tgt in self._ALLOWED_KEYS and tgt not in out:
                out[tgt] = v
        if (
            "investigation_target" not in out
            and isinstance(data.get("investigation_token"), str)
            and data["investigation_token"] in self._ALLOWED_TARGETS
        ):
            out["investigation_target"] = data["investigation_token"]
        if (
            "investigation_target" not in out
            and isinstance(data.get("investigation_signals"), list)
            and data["investigation_signals"]
            and isinstance(data["investigation_signals"][0], str)
            and data["investigation_signals"][0] in self._ALLOWED_TARGETS
        ):
            out["investigation_target"] = data["investigation_signals"][0]
        return out

    def _parse_and_validate(self, raw: str) -> AdReviewAction:
        text = _extract_json_text(raw)
        if not text:
            raise ValueError("empty model response")
        try:
            data = json.loads(text)
        except json.JSONDecodeError as exc:
            raise ValueError(f"model produced invalid JSON: {exc}") from exc
        if not isinstance(data, dict):
            raise ValueError(f"model output is not a JSON object: {type(data).__name__}")
        coerced = self._coerce_to_schema(data)
        return self.action_model.model_validate(coerced)


print("HFInvestigator class defined.")


HFInvestigator class defined.


## 6 · Training stack — rollout collector

TRL's `GRPOTrainer` consumes a HuggingFace `Dataset` whose rows have
at minimum a `prompt` column. CounterFeint's reward signal lives at
the *episode* level (the grader score, the Investigator's per-step
reward channel, the Auditor's track A reward) so we bridge the two:

  1. Run a full episode with an instrumented `HFInvestigator` (via
     `run_three_agent_episode`); the recorder snapshots every
     `(prompt, completion)` pair as the episode runs.
  2. After the episode finishes, distribute the Investigator's total
     episode reward evenly across its non-fallback turns and emit one
     `InvestigatorTrainingSample` per turn.


In [8]:
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional

from counterfeint.scripted import HeuristicAuditor, ReactiveFraudster


@dataclass
class InvestigatorTrainingSample:
    prompt: str
    completion: str
    reward: float
    task_id: str
    seed: int
    step_idx: int
    terminal_grader_score: float = 0.0
    end_reason: Optional[str] = None
    metadata: Dict[str, Any] = field(default_factory=dict)

    def to_dict(self) -> Dict[str, Any]:
        return {
            "prompt": self.prompt,
            "completion": self.completion,
            "reward": float(self.reward),
            "task_id": self.task_id,
            "seed": int(self.seed),
            "step_idx": int(self.step_idx),
            "terminal_grader_score": float(self.terminal_grader_score),
            "end_reason": self.end_reason,
            "metadata": dict(self.metadata),
        }


class RecordingHFInvestigator:
    """Decorator around HFInvestigator that records each act() call."""

    def __init__(self, inner: "HFInvestigator") -> None:
        self._inner = inner
        self.step_records: List[Dict[str, Any]] = []
        self._last_step_idx: int = 0

    def __getattr__(self, name: str) -> Any:
        return getattr(self._inner, name)

    @property
    def fallback_count(self) -> int:
        return getattr(self._inner, "fallback_count", 0)

    def reset(self) -> None:
        self.step_records.clear()
        self._last_step_idx = 0
        self._inner.reset()

    def act(self, observation: Dict[str, Any]) -> Any:
        result = self._inner.act(observation)
        self._last_step_idx += 1
        prompt = getattr(self._inner, "last_prompt", None)
        completion = getattr(self._inner, "last_completion", None)
        self.step_records.append({
            "step_idx": self._last_step_idx,
            "prompt": prompt,
            "completion": completion,
            "fallback_used": prompt is None or completion is None,
            "action_repr": repr(result),
        })
        return result


def _summarise_action(role: str, action: Any) -> str:
    """Compact one-line summary of any role's action for the live trace."""
    def _g(name: str, default: Any = None) -> Any:
        return getattr(action, name, None) if hasattr(action, name) else (
            action.get(name, default) if isinstance(action, dict) else default
        )

    at = _g("action_type", "?")
    ad = _g("ad_id", "")
    parts = [str(at)]
    if ad:
        parts.append(str(ad))
    if at == "investigate":
        tgt = _g("investigation_target")
        if tgt:
            parts.append(f"target={tgt}")
    elif at == "verdict":
        v = _g("verdict")
        c = _g("confidence")
        rationale = (_g("rationale") or "").strip()
        if v:
            parts.append(str(v))
        if c is not None:
            try:
                parts.append(f"@{float(c):.2f}")
            except (TypeError, ValueError):
                pass
        if rationale:
            if len(rationale) > TRACE_MAX_RATIONALE_CHARS:
                rationale = rationale[: TRACE_MAX_RATIONALE_CHARS - 3] + "..."
            parts.append(f'"{rationale}"')
    elif at == "link_accounts":
        linked = _g("linked_ad_id")
        reason = (_g("link_reason") or "").strip()
        if linked:
            parts.append(f"<-> {linked}")
        if reason:
            if len(reason) > TRACE_MAX_RATIONALE_CHARS:
                reason = reason[: TRACE_MAX_RATIONALE_CHARS - 3] + "..."
            parts.append(f'"{reason}"')
    elif at in {"propose_ad", "modify_pending_ad"}:
        cat = _g("category")
        if cat:
            parts.append(f"cat={cat}")
        copy = (_g("ad_copy") or _g("new_ad_copy") or "").strip()
        if copy:
            if len(copy) > TRACE_MAX_RATIONALE_CHARS:
                copy = copy[: TRACE_MAX_RATIONALE_CHARS - 3] + "..."
            parts.append(f'"{copy}"')
    return " ".join(parts)


class TracingPolicy:
    """Thin wrapper that prints one trace line per .act() and forwards the call."""

    _ROLE_TAG = {
        "fraudster":    "FRAUD ",
        "investigator": "INVEST",
        "auditor":      "AUDIT ",
    }

    def __init__(self, inner: Any, role: str, enabled: bool = True) -> None:
        self._inner = inner
        self._role = role
        self._enabled = bool(enabled)
        self._n = 0

    def __getattr__(self, name: str) -> Any:
        return getattr(self._inner, name)

    def reset(self) -> None:
        self._n = 0
        if hasattr(self._inner, "reset"):
            self._inner.reset()

    def act(self, observation: Dict[str, Any]) -> Any:
        result = self._inner.act(observation)
        self._n += 1
        if self._enabled:
            tag = self._ROLE_TAG.get(self._role, self._role.upper()[:6])
            inner_name = type(self._inner).__name__
            fallback = ""
            if isinstance(self._inner, RecordingHFInvestigator):
                rec = self._inner.step_records[-1] if self._inner.step_records else {}
                if rec.get("fallback_used"):
                    fallback = " [FB]"
                    inner_name = "HFInvestigator"
            elif getattr(self._inner, "last_error", None) and getattr(
                self._inner, "fallback_count", 0
            ) > 0:
                fallback = " [FB]"
            print(
                f"      {tag} #{self._n:02d} ({inner_name:<22}){fallback}  "
                f"{_summarise_action(self._role, result)}",
                flush=True,
            )
        return result


PolicyFactory = Callable[[], Any]


# Per-step shaping weights for the episode-end Investigator reward.  Verdicts
# and link_accounts are consequential decisions; investigates are preparatory.
# Splitting 80/20 (with matched counts → each verdict carries 4× the credit of
# each investigate) gives the Investigator a stronger gradient on the action
# that actually moves the grader, without dropping the credit on tool use.
# See ROUND_2_Q5_REALISM_REWARDS_TRAINING.md §3.2 + §5.1.
_VERDICT_REWARD_SHARE = 0.80
_VERDICT_ACTION_TYPES = ("verdict", "link_accounts")


def _classify_action(action_repr: Optional[str]) -> str:
    """Return ``"verdict"`` for consequential actions, ``"investigate"`` otherwise."""
    if not action_repr:
        return "investigate"
    text = action_repr.lower()
    return "verdict" if any(f"action_type='{t}'" in text for t in _VERDICT_ACTION_TYPES) else "investigate"


def _records_to_samples(
    records: List[Dict[str, Any]],
    *,
    episode_result: Dict[str, Any],
    task_id: str,
    seed: int,
) -> List[InvestigatorTrainingSample]:
    grader_score = float(episode_result.get("grader_score", 0.0))
    end_reason = episode_result.get("end_reason")
    rewards_by_role = episode_result.get("rewards_by_role") or {}
    investigator_total = float(rewards_by_role.get("investigator", 0.0))

    investigator_records = [
        r for r in records
        if r.get("prompt") is not None and r.get("completion") is not None
    ]
    if not investigator_records:
        logger.warning(
            "No usable Investigator turns in episode %s/seed=%s — every step "
            "fell back to the scripted policy.",
            task_id, seed,
        )
        return []

    classes = [_classify_action(r.get("action_repr")) for r in investigator_records]
    n_verdict = sum(1 for c in classes if c == "verdict")
    n_invest = len(investigator_records) - n_verdict

    if n_verdict == 0 or n_invest == 0:
        # All-one-type episode — fall back to uniform so we don't divide by zero
        # or starve the only existing action class of credit.
        per_turn = investigator_total / len(investigator_records)
        per_turn_rewards = [per_turn] * len(investigator_records)
    else:
        verdict_share = _VERDICT_REWARD_SHARE * investigator_total / n_verdict
        invest_share = (1.0 - _VERDICT_REWARD_SHARE) * investigator_total / n_invest
        per_turn_rewards = [
            verdict_share if c == "verdict" else invest_share for c in classes
        ]

    return [
        InvestigatorTrainingSample(
            prompt=r["prompt"],
            completion=r["completion"],
            reward=per_turn_rewards[i],
            task_id=task_id,
            seed=seed,
            step_idx=int(r["step_idx"]),
            terminal_grader_score=grader_score,
            end_reason=end_reason,
            metadata={
                "action_repr": r.get("action_repr"),
                "action_class": classes[i],
            },
        )
        for i, r in enumerate(investigator_records)
    ]


def collect_episode(
    *,
    hf_investigator: HFInvestigator,
    task_id: str,
    seed: int,
    fraudster_factory: Optional[PolicyFactory] = None,
    auditor_factory: Optional[PolicyFactory] = None,
    env_base_url: Optional[str] = None,
    log: bool = False,
    show_trace: bool = False,
) -> List[InvestigatorTrainingSample]:
    from counterfeint.inference import ENV_URL, run_three_agent_episode

    fraudster_factory = fraudster_factory or (lambda: ReactiveFraudster(seed=seed))
    auditor_factory = auditor_factory or (lambda: HeuristicAuditor())

    recorder = RecordingHFInvestigator(hf_investigator)
    recorder.reset()

    fraudster = fraudster_factory()
    investigator: Any = recorder
    auditor = auditor_factory()
    if show_trace:
        fraudster = TracingPolicy(fraudster, "fraudster", enabled=True)
        investigator = TracingPolicy(recorder, "investigator", enabled=True)
        auditor = TracingPolicy(auditor, "auditor", enabled=True)

    result = run_three_agent_episode(
        task_id,
        fraudster_policy=fraudster,
        investigator_policy=investigator,
        auditor_policy=auditor,
        env_base_url=env_base_url or ENV_URL,
        seed=seed,
        log=log,
    )

    return _records_to_samples(
        recorder.step_records,
        episode_result=result,
        task_id=task_id,
        seed=seed,
    )


def collect_dataset(
    *,
    hf_investigator: HFInvestigator,
    seeds_by_task: Dict[str, List[int]],
    fraudster_factory: Optional[PolicyFactory] = None,
    auditor_factory: Optional[PolicyFactory] = None,
    env_base_url: Optional[str] = None,
    show_trace: bool = False,
) -> List[InvestigatorTrainingSample]:
    out: List[InvestigatorTrainingSample] = []
    n_eps = sum(len(v) for v in seeds_by_task.values())
    done = 0
    skipped = 0
    for task_id, seeds in seeds_by_task.items():
        for seed in seeds:
            done += 1
            print(f"  [{done}/{n_eps}] {task_id} seed={seed} ...", flush=True)
            try:
                samples = collect_episode(
                    hf_investigator=hf_investigator,
                    task_id=task_id,
                    seed=seed,
                    fraudster_factory=fraudster_factory,
                    auditor_factory=auditor_factory,
                    env_base_url=env_base_url,
                    show_trace=show_trace,
                )
                out.extend(samples)
                if show_trace:
                    print(
                        f"      -> {len(samples)} usable Investigator turn(s) "
                        f"| fallback {hf_investigator.fallback_count}/"
                        f"{hf_investigator.call_count}",
                        flush=True,
                    )
            except Exception as exc:
                skipped += 1
                print(
                    f"      SKIPPED ({type(exc).__name__}: {exc}). "
                    f"Continuing with next seed.",
                    flush=True,
                )
    if skipped:
        print(
            f"\n  Note: {skipped}/{n_eps} episodes were skipped due to "
            f"transport errors (commonly Ollama timeouts under low-VRAM "
            f"conditions). Set USE_OLLAMA_FRAUDSTER=False or "
            f"LLM_FRAUDSTER_RATIO=0.0 in §1 to avoid them.",
            flush=True,
        )
    return out


def samples_to_hf_dataset(samples: List[InvestigatorTrainingSample]):
    from datasets import Dataset
    return Dataset.from_list([s.to_dict() for s in samples])


print("Rollout collector defined.")


Rollout collector defined.


## 7 · Training stack — GRPO config + opponent factory

`GRPOConfig` holds the placeholder hyperparameters; they're tuned by
hand at training time. `opponent_factory` honours `LLM_FRAUDSTER_RATIO`
from §1: the default 1.0 means every rollout uses the LLM Fraudster
(the strongest multi-agent story); set it to 0.0 to use only the
scripted `ReactiveFraudster` (much faster, weaker narrative).


In [9]:
import random

from counterfeint.agents import LLMFraudster


@dataclass
class GRPOConfig:
    learning_rate: float = 1e-6
    group_size: int = 4
    kl_coefficient: float = 0.01
    max_response_tokens: int = 512
    grad_accum_steps: int = 4
    eval_every_n_steps: int = 25
    seed: int = 7

    def to_dict(self) -> Dict[str, Any]:
        return {
            "learning_rate": self.learning_rate,
            "group_size": self.group_size,
            "kl_coefficient": self.kl_coefficient,
            "max_response_tokens": self.max_response_tokens,
            "grad_accum_steps": self.grad_accum_steps,
            "eval_every_n_steps": self.eval_every_n_steps,
            "seed": self.seed,
        }


GRPO_CONFIG = GRPOConfig()
fraudster_rng = random.Random(GRPO_CONFIG.seed)


def opponent_factory() -> Any:
    """Build a Fraudster opponent for one rollout, per LLM_FRAUDSTER_RATIO."""
    if not USE_OLLAMA_FRAUDSTER or LLM_FRAUDSTER_RATIO <= 0.0:
        return ReactiveFraudster(seed=fraudster_rng.randrange(0, 2**31))
    use_llm = fraudster_rng.random() < LLM_FRAUDSTER_RATIO
    if use_llm:
        return LLMFraudster(
            model_name=FRAUDSTER_OLLAMA_MODEL,
            api_base_url=os.environ["API_BASE_URL"],
            timeout_s=LLM_FRAUDSTER_TIMEOUT_S,
            fallback_seed=fraudster_rng.randrange(0, 2**31),
        )
    return ReactiveFraudster(seed=fraudster_rng.randrange(0, 2**31))


print("GRPO config:")
print(json.dumps(GRPO_CONFIG.to_dict(), indent=2))
print(f"\nOpponent factory: LLM_FRAUDSTER_RATIO={LLM_FRAUDSTER_RATIO}, "
      f"USE_OLLAMA_FRAUDSTER={USE_OLLAMA_FRAUDSTER}")


GRPO config:
{
  "learning_rate": 1e-06,
  "group_size": 4,
  "kl_coefficient": 0.01,
  "max_response_tokens": 512,
  "grad_accum_steps": 4,
  "eval_every_n_steps": 25,
  "seed": 7
}

Opponent factory: LLM_FRAUDSTER_RATIO=1.0, USE_OLLAMA_FRAUDSTER=True


## Preflight checks (run before §8)

This cell tests every prerequisite that §8 onwards depends on, **without
loading the heavy 6 GB model yet**. If anything fails here you'll see a
single clear error message instead of a 200-line `transformers` /
`bitsandbytes` traceback halfway through cell §8.

It checks:

1. **PyTorch + CUDA** — version, GPU name, free VRAM (warns if <14 GB).
2. **`BASE_MODEL` cache integrity** — total on-disk size (in `blobs/`
   on Linux/macOS, in `snapshots/` on Windows-without-symlink),
   presence of any `.incomplete` partial files (which would mean a
   frozen download), and expected files in the snapshot.
3. **FraudArena server** — `GET /health` against `COUNTERFEINT_ENV_URL`.
4. **Ollama** — `GET /api/tags` and confirms `FRAUDSTER_OLLAMA_MODEL`
   is pulled (skipped if `USE_OLLAMA_FRAUDSTER=False`).
5. **Output dirs** — `OUT_DIR` / `EVAL_OUT_DIR` are writable.

Re-run this cell anytime you're unsure of state — it's read-only
besides creating the output dirs.


In [10]:
import json as _json
import urllib.error
import urllib.request
from pathlib import Path as _Path

print("=" * 64)
print("PREFLIGHT CHECKS")
print("=" * 64)

_failures: list = []
_warnings: list = []

print("\n[1/5] PyTorch + GPU")
try:
    import torch as _torch

    print(f"      torch version    : {_torch.__version__}")
    _cuda_ok = _torch.cuda.is_available()
    print(f"      CUDA available   : {_cuda_ok}")
    if _cuda_ok:
        _props = _torch.cuda.get_device_properties(0)
        _free_b, _total_b = _torch.cuda.mem_get_info(0)
        print(f"      device           : {_props.name}")
        print(f"      VRAM total       : {_props.total_memory / 1e9:.1f} GB")
        print(f"      VRAM free now    : {_free_b / 1e9:.1f} GB")
        if _props.total_memory < 10e9 and not DRY_RUN and USE_OLLAMA_FRAUDSTER:
            _warnings.append(
                f"GPU has {_props.total_memory / 1e9:.1f} GB VRAM. "
                f"1.5B QLoRA + 8B Ollama on the same GPU usually needs >=10 GB. "
                f"Either set USE_OLLAMA_FRAUDSTER=False or MODE='smoke'."
            )
    elif not DRY_RUN:
        _failures.append(
            "No CUDA GPU detected. Real GRPO training needs a GPU. "
            "Set MODE='smoke' to skip the actual train() call."
        )
except ImportError:
    _failures.append("torch is not installed. Re-run the §2 install cell.")

print("\n[2/5] HuggingFace BASE_MODEL cache")
_hf_home = _Path(os.environ.get("HF_HOME", _Path.home() / ".cache" / "huggingface"))
_model_dir = _hf_home / "hub" / f"models--{BASE_MODEL.replace('/', '--')}"
print(f"      cache root       : {_hf_home}")
print(f"      model dir        : {_model_dir}")
if not _model_dir.exists():
    _dl_extra = ' --exclude "original/*"' if BASE_MODEL.startswith("meta-llama/") else ""
    _failures.append(
        f"BASE_MODEL cache dir does not exist. Download with:\n"
        f"        set HF_HUB_ENABLE_HF_TRANSFER=0\n"
        f"        python -m huggingface_hub.commands.huggingface_cli "
        f"download {BASE_MODEL}{_dl_extra}"
    )
else:
    _blobs_dir = _model_dir / "blobs"
    _snapshots_dir = _model_dir / "snapshots"

    _blob_files: list = []
    if _blobs_dir.exists():
        _blob_files = [f for f in _blobs_dir.iterdir() if f.is_file()]
    _incomplete = [f for f in _blob_files if f.suffix == ".incomplete"]
    _complete_blobs = [f for f in _blob_files if f.suffix != ".incomplete"]
    _blob_gb = sum(f.stat().st_size for f in _complete_blobs) / 1e9

    _snap_files: list = []
    _snap = None
    if _snapshots_dir.exists():
        _snap_subdirs = [d for d in _snapshots_dir.iterdir() if d.is_dir()]
        if _snap_subdirs:
            _snap = _snap_subdirs[0]
            _snap_files = [f for f in _snap.iterdir() if f.is_file() or f.is_symlink()]
    _snap_real_gb = sum(
        f.stat().st_size for f in _snap_files if f.is_file() and not f.is_symlink()
    ) / 1e9
    _on_disk_gb = max(_blob_gb, _snap_real_gb)

    print(f"      complete blobs   : {len(_complete_blobs)}  ({_blob_gb:.2f} GB)")
    print(f"      snapshot files   : {len(_snap_files)}  ({_snap_real_gb:.2f} GB real)")
    print(f"      .incomplete files: {len(_incomplete)}")
    if _blob_gb == 0 and _snap_real_gb > 0:
        print("      layout           : Windows-no-symlink (snapshot/ holds real files - OK)")

    _expected_gb = (
        6.4 if "Llama-3.2-3B" in BASE_MODEL
        else 3.1 if "Qwen2.5-1.5B" in BASE_MODEL
        else 6.2 if "Qwen2.5-3B" in BASE_MODEL
        else None
    )
    if _incomplete:
        _failures.append(
            f"BASE_MODEL has {len(_incomplete)} .incomplete file(s) - "
            f"download was interrupted. Re-run the download command "
            f"(it will resume from where it left off)."
        )
    if _expected_gb is not None and _on_disk_gb < _expected_gb * 0.85:
        _failures.append(
            f"BASE_MODEL on-disk total {_on_disk_gb:.2f} GB "
            f"(expected ~{_expected_gb:.1f} GB). Download is incomplete."
        )

    if _snap is not None:
        _present = {f.name for f in _snap_files}
        _required = {"config.json", "tokenizer_config.json"}
        _missing = _required - _present
        _has_weights = (
            "model.safetensors" in _present
            or "model.safetensors.index.json" in _present
            or any(n.startswith("model-") and n.endswith(".safetensors") for n in _present)
        )
        if _missing:
            _failures.append(f"BASE_MODEL snapshot missing: {sorted(_missing)}")
        if not _has_weights:
            _failures.append(
                "BASE_MODEL snapshot has no model.safetensors / shards / index file."
            )
    elif not _snapshots_dir.exists():
        _failures.append(f"snapshots/ missing under {_model_dir}.")

print("\n[3/5] FraudArena server")
_env_url = os.environ.get("COUNTERFEINT_ENV_URL", "http://localhost:8000")
try:
    with urllib.request.urlopen(f"{_env_url}/health", timeout=3) as _r:
        if _r.status == 200:
            print(f"      {_env_url}/health -> 200 OK")
        else:
            _failures.append(f"FraudArena /health returned {_r.status}")
except Exception as _exc:
    _failures.append(
        f"FraudArena unreachable at {_env_url} "
        f"({type(_exc).__name__}: {_exc}). Re-run §3."
    )

print(f"\n[4/5] Ollama Fraudster (USE_OLLAMA_FRAUDSTER={USE_OLLAMA_FRAUDSTER})")
if USE_OLLAMA_FRAUDSTER:
    _api_base = os.environ.get("API_BASE_URL", "http://localhost:11434/v1").rstrip("/")
    _root = _api_base[:-3] if _api_base.endswith("/v1") else _api_base
    _tags_url = f"{_root}/api/tags"
    try:
        with urllib.request.urlopen(_tags_url, timeout=3) as _r:
            _data = _json.loads(_r.read().decode("utf-8"))
        _names = [m.get("name", "") for m in _data.get("models", [])]
        print(f"      tags endpoint    : {_tags_url}")
        print(f"      models served    : {_names}")
        _stem = FRAUDSTER_OLLAMA_MODEL.split(":")[0]
        if not any(n.startswith(_stem) for n in _names):
            _failures.append(
                f"Ollama is up but {FRAUDSTER_OLLAMA_MODEL} is not pulled. "
                f"Run: ollama pull {FRAUDSTER_OLLAMA_MODEL}"
            )
    except Exception as _exc:
        _failures.append(
            f"Ollama unreachable ({type(_exc).__name__}: {_exc}). Re-run §4."
        )
else:
    print("      skipped (USE_OLLAMA_FRAUDSTER=False)")

print("\n[5/5] Output directories")
for _d in (OUT_DIR, EVAL_OUT_DIR):
    try:
        _d.mkdir(parents=True, exist_ok=True)
        _probe = _d / ".write_probe"
        _probe.write_text("ok", encoding="utf-8")
        _probe.unlink()
        print(f"      writable        : {_d.resolve()}")
    except Exception as _exc:
        _failures.append(f"{_d} not writable: {_exc}")

print("\n" + "=" * 64)
if _failures:
    print(f"FAIL  ({len(_failures)} issue{'s' if len(_failures) > 1 else ''})")
    for _i, _msg in enumerate(_failures, 1):
        print(f"  {_i}. {_msg}")
    print("=" * 64)
    raise RuntimeError("Preflight failed. Fix the issues above before §8 onwards.")
elif _warnings:
    print(f"PASS WITH WARNINGS  ({len(_warnings)})")
    for _i, _msg in enumerate(_warnings, 1):
        print(f"  {_i}. {_msg}")
    print("\nProceed only if you understand the warnings.")
else:
    print("PASS - all checks green. Proceed to §8.")
print("=" * 64)


PREFLIGHT CHECKS

[1/5] PyTorch + GPU
      torch version    : 2.9.1+cu128
      CUDA available   : True
      device           : NVIDIA GeForce RTX 5060 Laptop GPU
      VRAM total       : 8.5 GB
      VRAM free now    : 7.3 GB

[2/5] HuggingFace BASE_MODEL cache
      cache root       : C:\Users\abhij\.cache\huggingface
      model dir        : C:\Users\abhij\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct
      complete blobs   : 0  (0.00 GB)
      snapshot files   : 10  (3.10 GB real)
      .incomplete files: 0
      layout           : Windows-no-symlink (snapshot/ holds real files - OK)

[3/5] FraudArena server
      http://localhost:8000/health -> 200 OK

[4/5] Ollama Fraudster (USE_OLLAMA_FRAUDSTER=True)
      tags endpoint    : http://localhost:11434/api/tags
      models served    : ['llama3.1:8b']

[5/5] Output directories
      writable        : C:\Users\abhij\OneDrive\Desktop\Brain Labs\Coding\OpenEnv Hackathon\counterfeint\training_outputs\investigator_grpo_gen0

## 8 · Load `BASE_MODEL` with QLoRA + LoRA adapter

Loads `Qwen/Qwen2.5-1.5B-Instruct` (Apache 2.0) in 4-bit
(BitsAndBytes nf4 + double-quant + bfloat16 compute) and attaches a
LoRA adapter (`r=16`, `alpha=32`, `q/k/v/o_proj`). The result is a
~1.2 GB VRAM footprint with ~2.4 M trainable parameters — cleanly fits
on a 12 GB consumer GPU (RTX 3060 / 4060) alongside an 8 B Ollama
Fraudster, and leaves comfortable headroom on a Colab T4 (16 GB).

If you set `MODE='smoke'` and don't have CUDA, this cell will fail at
the BitsAndBytes step. That's intended: GRPO needs a GPU. The
`dry_run=True` skip in §10 only avoids the actual `trainer.train()`
call; the model still has to load to support the rollout-collection
sanity check.


In [11]:
print(f"Loading {BASE_MODEL} in 4-bit (this may take a couple of minutes) ...")
hf_investigator = HFInvestigator.from_pretrained(BASE_MODEL, load_in_4bit=True)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(hf_investigator.model)
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_cfg)
hf_investigator.model = model
model.print_trainable_parameters()

import time as _time
import torch as _torch

_first_param = next(model.parameters())
print(f"\nGPU sanity check:")
print(f"  trainable param dtype : {_first_param.dtype}")
print(f"  trainable param device: {_first_param.device}")
if _first_param.device.type == "cuda":
    _free_b, _total_b = _torch.cuda.mem_get_info(0)
    print(f"  VRAM free / total     : {_free_b/1e9:.2f} / {_total_b/1e9:.2f} GB")
    print(f"  VRAM allocated by us  : {_torch.cuda.memory_allocated(0)/1e9:.2f} GB")
else:
    print("  WARNING: model parameters are on CPU. Generation will be ~10x slower.")
    print("           If you have CUDA, restart kernel and re-run §8.")

_probe_messages = [
    {"role": "system", "content": "You output one line."},
    {"role": "user", "content": "Reply with exactly: pong"},
]
_probe_inputs = hf_investigator.tokenizer.apply_chat_template(
    _probe_messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
)
_probe_inputs = {k: v.to(_first_param.device) for k, v in _probe_inputs.items()}
_t0 = _time.perf_counter()
with _torch.inference_mode():
    _probe_out = model.generate(
        **_probe_inputs,
        max_new_tokens=8,
        do_sample=False,
        pad_token_id=hf_investigator.tokenizer.eos_token_id,
    )
_dt = _time.perf_counter() - _t0
_probe_text = hf_investigator.tokenizer.decode(
    _probe_out[0][_probe_inputs["input_ids"].shape[-1]:], skip_special_tokens=True
).strip()
print(f"  test generate (8 tok) : {_dt*1000:.0f} ms  -> {_probe_text!r}")
print(f"  expected on a 6-12 GB GPU: <800 ms; on CPU: 8-30 s")


Loading Qwen/Qwen2.5-1.5B-Instruct in 4-bit (this may take a couple of minutes) ...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815

GPU sanity check:
  trainable param dtype : torch.float32
  trainable param device: cuda:0
  VRAM free / total     : 4.74 / 8.55 GB
  VRAM allocated by us  : 1.64 GB
  test generate (8 tok) : 1340 ms  -> 'Pong!'
  expected on a 6-12 GB GPU: <800 ms; on CPU: 8-30 s


## 9 · Collect rollout samples

Runs `TRAINING_SEEDS` episodes against `opponent_factory` (the 100 %
LLM Fraudster by default). Each Investigator turn becomes one
`(prompt, completion, reward)` row; rows are flattened across
episodes and handed to TRL's `GRPOTrainer` in the next cell.

**Watch the `[ {n}/{total} ]` log** — if every episode reports
`fallback_count` ≈ episode-length the LLM model isn't producing
parseable JSON. Most common culprit: temperature too high or the
chat template missing — fix the model load above and re-run from §8.


In [12]:
print(f"Collecting {n_train_eps} rollout episodes (mode={MODE}) ...")
print(f"  Fraudster: {FRAUDSTER_OLLAMA_MODEL if USE_OLLAMA_FRAUDSTER else 'ReactiveFraudster'}")

samples = collect_dataset(
    hf_investigator=hf_investigator,
    seeds_by_task=TRAINING_SEEDS,
    fraudster_factory=opponent_factory,
    env_base_url=os.environ["COUNTERFEINT_ENV_URL"],
    show_trace=SHOW_INTERACTION_TRACE,
)

print(f"\nCollected {len(samples)} (prompt, completion, reward) samples.")
print(f"Investigator fallback_count this round: {hf_investigator.fallback_count}")

if samples:
    rewards = [s.reward for s in samples]
    print(f"Reward stats: min={min(rewards):.3f}  max={max(rewards):.3f}  "
          f"mean={sum(rewards)/len(rewards):.3f}")


  Fraudster: llama3.1:8b
  [1/6] task_1 seed=0 ...
      SKIPPED (ConnectionClosedError: received 1011 (internal error) keepalive ping timeout; then sent 1011 (internal error) keepalive ping timeout). Continuing with next seed.
  [2/6] task_1 seed=1 ...
      SKIPPED (ConnectionClosedError: received 1011 (internal error) keepalive ping timeout; then sent 1011 (internal error) keepalive ping timeout). Continuing with next seed.
  [3/6] task_2 seed=100 ...
      SKIPPED (ConnectionClosedError: received 1011 (internal error) keepalive ping timeout; then sent 1011 (internal error) keepalive ping timeout). Continuing with next seed.
  [4/6] task_2 seed=101 ...
      SKIPPED (ConnectionClosedError: received 1011 (internal error) keepalive ping timeout; then sent 1011 (internal error) keepalive ping timeout). Continuing with next seed.
  [5/6] task_3 seed=200 ...
      SKIPPED (ConnectionClosedError: received 1011 (internal error) keepalive ping timeout; then sent 1011 (internal error) keepal

## 10 · Run GRPO training

Hands the collected samples to TRL's `GRPOTrainer`. The reward
function is a closure over a precomputed lookup
`{(prompt, completion): reward}` so we never re-run the FraudArena
env from inside the trainer's hot loop. GRPO uses *relative*
advantage so the small negative default for never-seen
(prompt, completion) pairs is fine.

In `smoke` mode (`DRY_RUN=True`) we skip `trainer.train()` and write
a placeholder adapter directory; the eval cell below will fall back
to the scripted Investigator for the `after` sweep.


In [ ]:
import hashlib
from typing import List as _List

OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "training_config.json").write_text(
    json.dumps({
        "base_model": BASE_MODEL,
        "fraudster_model": FRAUDSTER_OLLAMA_MODEL if USE_OLLAMA_FRAUDSTER else "scripted",
        "llm_fraudster_ratio": LLM_FRAUDSTER_RATIO,
        "grpo_config": GRPO_CONFIG.to_dict(),
        "seeds_by_task": TRAINING_SEEDS,
        "dry_run": DRY_RUN,
        "n_samples": len(samples),
    }, indent=2),
    encoding="utf-8",
)


def _sample_key(prompt: str, completion: str) -> str:
    h = hashlib.sha256()
    h.update(prompt.encode("utf-8", errors="replace"))
    h.update(b"\x00")
    h.update(completion.encode("utf-8", errors="replace"))
    return h.hexdigest()


def make_reward_fn(sample_lookup: Dict[str, float]):
    def reward_fn(prompts: _List[str], completions: _List[str], **_: Any) -> _List[float]:
        return [
            float(sample_lookup.get(_sample_key(p, c), -0.01))
            for p, c in zip(prompts, completions)
        ]
    return reward_fn


if not samples:
    raise RuntimeError(
        "No training samples collected — every episode fell back to the "
        "scripted policy. Check the FraudArena server log + GPU memory."
    )

if DRY_RUN:
    print("DRY_RUN=True (smoke mode) — skipping GRPOTrainer.train().")
    adapter_path = OUT_DIR / "lora_adapter_dry_run"
    adapter_path.mkdir(exist_ok=True)
    print(f"Placeholder adapter dir: {adapter_path}")
else:
    from trl import GRPOConfig as TRLGRPOConfig
    from trl import GRPOTrainer

    dataset = samples_to_hf_dataset(samples)
    sample_lookup = {_sample_key(s.prompt, s.completion): s.reward for s in samples}
    reward_fn = make_reward_fn(sample_lookup)

    trl_cfg = TRLGRPOConfig(
        output_dir=str(OUT_DIR / "trl_checkpoints"),
        learning_rate=GRPO_CONFIG.learning_rate,
        num_generations=GRPO_CONFIG.group_size,
        max_completion_length=GRPO_CONFIG.max_response_tokens,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=GRPO_CONFIG.grad_accum_steps,
        beta=GRPO_CONFIG.kl_coefficient,
        seed=GRPO_CONFIG.seed,
        logging_steps=5,
        save_steps=GRPO_CONFIG.eval_every_n_steps,
        report_to="none",
    )

    trainer = GRPOTrainer(
        model=model,
        args=trl_cfg,
        reward_funcs=[reward_fn],
        train_dataset=dataset,
        processing_class=hf_investigator.tokenizer,
    )
    print("Starting GRPOTrainer.train() ...")
    trainer.train()

    adapter_path = OUT_DIR / "lora_adapter"
    model.save_pretrained(str(adapter_path))
    hf_investigator.tokenizer.save_pretrained(str(adapter_path))
    print(f"\nSaved LoRA adapter: {adapter_path}")


## 11 · Before-vs-after held-out eval

Both the `before` (scripted Investigator) and `after` (trained LoRA
adapter on top of `BASE_MODEL`) sweeps run against the **same**
`ReactiveFraudster(seed=42)` opponent over the held-out
`EVAL_SEEDS` (10 per task, disjoint from `TRAINING_SEEDS`).

> The eval Fraudster is `ReactiveFraudster(seed=42)` — deterministic
> by design — so before/after numbers are stable across re-runs. The
> *training* curriculum used the LLM Fraudster, so the trained
> Investigator has been hardened against a stronger adversary than
> the eval distribution presents — that's the point.

In `smoke` mode the `after` factory falls back to the scripted policy
since training was skipped; the artefacts are still produced so you
can verify the eval lane works.


In [ ]:
from counterfeint.eval_suite import run_before_after
from counterfeint.scripted import ScriptedInvestigator


def _scripted_factory():
    return ScriptedInvestigator()


if DRY_RUN:
    print("smoke mode → after_investigator_factory = ScriptedInvestigator (no adapter trained).")
    after_factory = _scripted_factory
    after_tag = f"{TRAINED_TAG}_smoke"
else:
    _adapter_path = str(adapter_path)

    def _after_factory():
        return HFInvestigator.from_pretrained(
            BASE_MODEL,
            load_in_4bit=True,
            lora_path=_adapter_path,
        )

    after_factory = _after_factory
    after_tag = TRAINED_TAG

summary = run_before_after(
    before_tag="scripted_baseline",
    after_tag=after_tag,
    before_investigator_factory=_scripted_factory,
    after_investigator_factory=after_factory,
    out_dir=EVAL_OUT_DIR,
    seeds=EVAL_SEEDS_OVERRIDE,
    env_base_url=os.environ["COUNTERFEINT_ENV_URL"],
)

print(json.dumps(summary["before"], indent=2))
print(json.dumps(summary["after"], indent=2))


In [ ]:
from IPython.display import Image, Markdown, display

plot_path = EVAL_OUT_DIR / "eval_plot.png"
summary_md = EVAL_OUT_DIR / "eval_summary.md"

if plot_path.exists():
    display(Image(filename=str(plot_path)))
else:
    print(f"(no plot at {plot_path} — matplotlib not installed?)")

if summary_md.exists():
    display(Markdown(summary_md.read_text(encoding="utf-8")))
else:
    print(f"(no summary md at {summary_md})")


## 12 · Cleanup

Stop the FraudArena server and Ollama. Safe to re-run multiple times —
the calls are idempotent.


In [ ]:
for var_name, label in (("server_proc", "FraudArena"), ("ollama_proc", "Ollama")):
    proc = globals().get(var_name)
    if proc is None:
        continue
    try:
        proc.terminate()
        proc.wait(timeout=5)
        print(f"{label} stopped (exit code {proc.returncode}).")
    except Exception as exc:
        print(f"{label}: {exc}")

print("\nDone. Headline artefacts:")
for p in (EVAL_OUT_DIR / "eval_results.json",
          EVAL_OUT_DIR / "eval_summary.md",
          EVAL_OUT_DIR / "eval_plot.png"):
    if p.exists():
        print(f"  {p.resolve()}  ({p.stat().st_size} bytes)")
    else:
        print(f"  {p.resolve()}  (MISSING)")
